# 06 — Grad-CAM Explainability Analysis
**WikiArt Painting Classifier** — NOVA IMS Deep Learning 2025/2026

This notebook applies **Grad-CAM** (Gradient-weighted Class Activation Mapping) to the best-performing model to understand which regions of a painting drive the classification decision.

We analyse:
1. **Correct predictions** — confirming the model attends to semantically meaningful features (brushwork, composition, colour palette)
2. **Misclassified samples** — diagnosing failure modes and potential dataset biases
3. **Cross-class comparisons** — visualising what distinguishes similar artists in the model’s representation

All heatmaps are saved to `results/gradcam/` for the report.

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath(".."))
from src.data_loader import build_datasets
from src.gradcam import (
    make_gradcam_heatmap,
    make_gradcam_heatmap_vit,
    overlay_heatmap,
    save_gradcam,
    find_last_conv_layer,
    find_vit_backbone_layer,
    generate_gradcam_for_dataset,
)

FIGURES_DIR = "../results/figures"
GRADCAM_DIR = "../results/gradcam"
os.makedirs(GRADCAM_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

## 1. Configuration & Data Loading

In [ ]:
# Best model — ViT-B/16 achieved the highest F1-macro (0.850) in 07_evaluation
BEST_MODEL_NAME = "vit"
BEST_MODEL_PATH = "../results/models/vit.keras"
IS_VIT = True

IMG_SIZE = (224, 224)
BATCH_SIZE = 16

In [ ]:
# Load test dataset
_, _, test_ds, NUM_CLASSES = build_datasets(
    splits_dir="../data/splits",
    img_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    augment_train=False,
    use_processed=True,
    processed_dir="../data/processed",
)

test_df = pd.read_csv("../data/splits/test.csv", encoding="utf-8-sig")
artist_names = sorted(test_df["artist"].unique())
print(f"Test set: {len(test_df)} images, {NUM_CLASSES} classes")

## 2. Load Best Model

In [ ]:
# Custom objects needed to load models:
# - transfer: Lambda wrapping resnet50.preprocess_input
# - vit: CosineWarmup LR schedule in AdamW optimizer
from src.models.vit import CosineWarmup

custom_objects = {
    "preprocess_input": tf.keras.applications.resnet50.preprocess_input,
    "CosineWarmup": CosineWarmup,
}

with tf.keras.utils.custom_object_scope(custom_objects):
    model = tf.keras.models.load_model(BEST_MODEL_PATH)

print(f"Model: {BEST_MODEL_NAME}")
print(f"Parameters: {model.count_params():,}")

if IS_VIT:
    backbone_name = find_vit_backbone_layer(model)
    print(f"ViT backbone layer: {backbone_name}")
else:
    conv_name = find_last_conv_layer(model)
    print(f"Last conv layer: {conv_name}")

## 3. Grad-CAM on Individual Samples
Visualise heatmaps for a few hand-picked test images to build intuition.

In [ ]:
# Collect a batch of test images with their labels
sample_images, sample_labels = next(iter(test_ds))
print(f"Batch shape: {sample_images.shape}, Labels: {sample_labels.shape}")

# Generate predictions
preds = model.predict(sample_images, verbose=0)
pred_classes = np.argmax(preds, axis=1)
true_classes = sample_labels.numpy()

In [ ]:
# Show Grad-CAM for first 8 samples
n_show = min(8, sample_images.shape[0])
fig, axes = plt.subplots(2, n_show, figsize=(3 * n_show, 6))

for i in range(n_show):
    img = sample_images[i:i+1]
    true_label = int(true_classes[i])

    if IS_VIT:
        heatmap, pred_label, conf = make_gradcam_heatmap_vit(model, img)
    else:
        heatmap, pred_label, conf = make_gradcam_heatmap(model, img)

    superimposed = overlay_heatmap(img[0].numpy(), heatmap)
    correct = true_label == pred_label
    color = "green" if correct else "red"

    # Original
    axes[0, i].imshow(img[0].numpy())
    axes[0, i].set_title(f"True: {artist_names[true_label]}", fontsize=7)
    axes[0, i].axis("off")

    # Grad-CAM overlay
    axes[1, i].imshow(superimposed)
    axes[1, i].set_title(f"Pred: {artist_names[pred_label]}\n({conf:.0%})", fontsize=7, color=color)
    axes[1, i].axis("off")

plt.suptitle(f"Grad-CAM — {BEST_MODEL_NAME} (top: original, bottom: heatmap overlay)", fontsize=12)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/gradcam_sample_overview.png", dpi=300, bbox_inches="tight")
plt.show()

## 4. Correct Predictions
Verify the model focuses on meaningful visual features for correctly classified paintings.

In [ ]:
# Find correctly classified samples
correct_indices = []
correct_images = []
correct_labels = []

for images, labels in test_ds:
    batch_preds = np.argmax(model.predict(images, verbose=0), axis=1)
    batch_true = labels.numpy()
    for i in range(len(batch_true)):
        if batch_preds[i] == batch_true[i]:
            correct_indices.append(len(correct_images))
            correct_images.append(images[i].numpy())
            correct_labels.append(int(batch_true[i]))
        if len(correct_images) >= 200:
            break
    if len(correct_images) >= 200:
        break

print(f"Collected {len(correct_images)} correct predictions")

# Pick diverse artists for display
seen_artists = set()
diverse_correct = []
for idx in range(len(correct_images)):
    label = correct_labels[idx]
    if label not in seen_artists:
        seen_artists.add(label)
        diverse_correct.append(idx)
    if len(diverse_correct) >= 8:
        break

fig, axes = plt.subplots(2, len(diverse_correct), figsize=(3 * len(diverse_correct), 6))
for col, idx in enumerate(diverse_correct):
    img = correct_images[idx][np.newaxis, ...]
    true_label = correct_labels[idx]

    if IS_VIT:
        heatmap, pred_label, conf = make_gradcam_heatmap_vit(model, img)
    else:
        heatmap, pred_label, conf = make_gradcam_heatmap(model, img)

    superimposed = overlay_heatmap(correct_images[idx], heatmap)

    axes[0, col].imshow(correct_images[idx])
    axes[0, col].set_title(artist_names[true_label], fontsize=7)
    axes[0, col].axis("off")

    axes[1, col].imshow(superimposed)
    axes[1, col].set_title(f"{conf:.0%}", fontsize=7, color="green")
    axes[1, col].axis("off")

plt.suptitle(f"Grad-CAM — Correct Predictions ({BEST_MODEL_NAME})", fontsize=12)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/gradcam_correct_predictions.png", dpi=300, bbox_inches="tight")
plt.show()

## 5. Misclassified Samples
Diagnose failure modes by examining what the model attends to when it makes mistakes.

In [ ]:
# Find misclassified samples
wrong_images = []
wrong_true = []
wrong_pred = []
wrong_conf = []

for images, labels in test_ds:
    batch_preds_probs = model.predict(images, verbose=0)
    batch_preds = np.argmax(batch_preds_probs, axis=1)
    batch_true = labels.numpy()
    for i in range(len(batch_true)):
        if batch_preds[i] != batch_true[i]:
            wrong_images.append(images[i].numpy())
            wrong_true.append(int(batch_true[i]))
            wrong_pred.append(int(batch_preds[i]))
            wrong_conf.append(float(batch_preds_probs[i, batch_preds[i]]))
        if len(wrong_images) >= 100:
            break
    if len(wrong_images) >= 100:
        break

print(f"Collected {len(wrong_images)} misclassified samples")

# Show the top-8 most confident wrong predictions (overconfident errors)
conf_order = np.argsort(wrong_conf)[::-1][:8]

fig, axes = plt.subplots(2, len(conf_order), figsize=(3 * len(conf_order), 6))
for col, idx in enumerate(conf_order):
    img = wrong_images[idx][np.newaxis, ...]
    true_label = wrong_true[idx]

    if IS_VIT:
        heatmap, pred_label, conf = make_gradcam_heatmap_vit(model, img)
    else:
        heatmap, pred_label, conf = make_gradcam_heatmap(model, img)

    superimposed = overlay_heatmap(wrong_images[idx], heatmap)

    axes[0, col].imshow(wrong_images[idx])
    axes[0, col].set_title(f"True: {artist_names[true_label]}", fontsize=7)
    axes[0, col].axis("off")

    axes[1, col].imshow(superimposed)
    axes[1, col].set_title(f"Pred: {artist_names[pred_label]}\n({conf:.0%})", fontsize=7, color="red")
    axes[1, col].axis("off")

plt.suptitle(f"Grad-CAM — Most Confident Misclassifications ({BEST_MODEL_NAME})", fontsize=12)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/gradcam_misclassifications.png", dpi=300, bbox_inches="tight")
plt.show()

## 6. Cross-Class Comparison
Compare what the model focuses on for artists with similar styles (e.g., Impressionists).

In [ ]:
# Pick pairs of frequently confused artists
# Use the confusion matrix from the evaluation to find top confused pairs
from sklearn.metrics import confusion_matrix as sk_cm

y_true_all, y_pred_all = [], []
for images, labels in test_ds:
    batch_preds = np.argmax(model.predict(images, verbose=0), axis=1)
    y_true_all.extend(labels.numpy())
    y_pred_all.extend(batch_preds)

y_true_all = np.array(y_true_all)
y_pred_all = np.array(y_pred_all)

cm = sk_cm(y_true_all, y_pred_all)
np.fill_diagonal(cm, 0)

# Top 3 most confused pairs
n_pairs = 3
flat_idx = np.argsort(cm.ravel())[::-1][:n_pairs]
rows, cols = np.unravel_index(flat_idx, cm.shape)

print("Most confused artist pairs:")
for r, c in zip(rows, cols):
    print(f"  {artist_names[r]} \u2192 {artist_names[c]} ({cm[r, c]} samples)")

In [ ]:
# For the top confused pair, show Grad-CAM side by side
pair_true = int(rows[0])
pair_pred = int(cols[0])
print(f"\nAnalysing: {artist_names[pair_true]} (misclassified as {artist_names[pair_pred]})")

# Find samples of this confusion
pair_images = []
for images, labels in test_ds:
    batch_preds = np.argmax(model.predict(images, verbose=0), axis=1)
    batch_true = labels.numpy()
    for i in range(len(batch_true)):
        if batch_true[i] == pair_true and batch_preds[i] == pair_pred:
            pair_images.append(images[i].numpy())
        if len(pair_images) >= 4:
            break
    if len(pair_images) >= 4:
        break

# Also find correctly classified samples of the predicted class for comparison
compare_images = []
for images, labels in test_ds:
    batch_preds = np.argmax(model.predict(images, verbose=0), axis=1)
    batch_true = labels.numpy()
    for i in range(len(batch_true)):
        if batch_true[i] == pair_pred and batch_preds[i] == pair_pred:
            compare_images.append(images[i].numpy())
        if len(compare_images) >= 4:
            break
    if len(compare_images) >= 4:
        break

n_show = min(4, len(pair_images), len(compare_images))
if n_show > 0:
    fig, axes = plt.subplots(2, n_show, figsize=(4 * n_show, 8))
    if n_show == 1:
        axes = axes[:, np.newaxis]

    for col in range(n_show):
        # Misclassified sample
        img = pair_images[col][np.newaxis, ...]
        if IS_VIT:
            heatmap, _, conf = make_gradcam_heatmap_vit(model, img)
        else:
            heatmap, _, conf = make_gradcam_heatmap(model, img)
        superimposed = overlay_heatmap(pair_images[col], heatmap)
        axes[0, col].imshow(superimposed)
        axes[0, col].set_title(f"True: {artist_names[pair_true]}\nPred: {artist_names[pair_pred]}", fontsize=8, color="red")
        axes[0, col].axis("off")

        # Correctly classified sample of the confused-with class
        img2 = compare_images[col][np.newaxis, ...]
        if IS_VIT:
            heatmap2, _, conf2 = make_gradcam_heatmap_vit(model, img2)
        else:
            heatmap2, _, conf2 = make_gradcam_heatmap(model, img2)
        superimposed2 = overlay_heatmap(compare_images[col], heatmap2)
        axes[1, col].imshow(superimposed2)
        axes[1, col].set_title(f"True & Pred: {artist_names[pair_pred]}", fontsize=8, color="green")
        axes[1, col].axis("off")

    plt.suptitle(
        f"Cross-Class Comparison: {artist_names[pair_true]} vs {artist_names[pair_pred]}\n"
        f"Top row: {artist_names[pair_true]} misclassified as {artist_names[pair_pred]}\n"
        f"Bottom row: Correctly classified {artist_names[pair_pred]}",
        fontsize=11
    )
    plt.tight_layout()
    plt.savefig(f"{FIGURES_DIR}/gradcam_cross_class_comparison.png", dpi=300, bbox_inches="tight")
    plt.show()
else:
    print("Not enough samples found for this pair.")

## 7. Batch Generation
Generate Grad-CAM heatmaps for all test set classes and save to `results/gradcam/`.

In [ ]:
# Reload test dataset (iterator may be exhausted)
_, _, test_ds_fresh, _ = build_datasets(
    splits_dir="../data/splits",
    img_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    augment_train=False,
    use_processed=True,
    processed_dir="../data/processed",
)

total = generate_gradcam_for_dataset(
    model, test_ds_fresh, artist_names, GRADCAM_DIR,
    is_vit=IS_VIT,
    max_correct=2,
    max_wrong=2,
)
print(f"\nTotal heatmaps saved: {total}")

## 8. Summary

Grad-CAM was applied to the best-performing model (**ViT-B/16**, F1-macro ≈ 0.82) across the full test set, producing **91 heatmaps** (46 correct, 45 misclassified) saved to `results/gradcam/`.

### Key findings

**Correct predictions:**
- For highly distinctive artists like **Gustave Doré** (97.7% confidence), the model focuses on the characteristic engraving line-work and dramatic chiaroscuro shading that defines his style.
- For **Albrecht Dürer**, attention concentrates on fine cross-hatching and detailed figure contours — the hallmarks of Northern Renaissance draughtsmanship.
- Landscape painters like **Vincent van Gogh** and **Martiros Saryan** show activation spread across broad colour fields and textural brushwork regions, consistent with their expressive impasto techniques.

**Misclassifications:**
- The most confident errors involve **Impressionist and Post-Impressionist painters** who share similar colour palettes and plein-air compositions:
  - **Claude Monet → Camille Pissarro** (most confused pair): both feature soft outdoor scenes with diffused light. The heatmaps show the model attending to similar landscape structures and colour harmonies in both, unable to distinguish Monet's looser, more atmospheric brushwork from Pissarro's more structured pointillist touches.
  - **Claude Monet → Childe Hassam**: American Impressionism closely mirrors French Impressionism, and the heatmaps confirm the model fixates on shared compositional elements (urban scenes, dappled light).
- **Salvador Dalí → Pablo Picasso** confusions (67.7% confidence) show the model attending to abstract figurative forms and bold colour contrasts — features shared across early-to-mid 20th-century modernist movements.
- **Ivan Shishkin** misclassifications show the model focusing on forest canopy and foliage textures, which overlap with several other Russian landscape painters in the dataset.

**Cross-class comparison (Claude Monet vs Camille Pissarro):**
- When Monet paintings are misclassified as Pissarro, the heatmaps highlight broad landscape compositions with natural colour gradients — features common to both artists.
- Correctly classified Pissarro works show similar activation patterns, confirming the model has not learned a strong discriminative boundary between these two Impressionists.
- This suggests the dataset may benefit from more samples of these closely related artists, or that style-level features alone are insufficient to separate painters within the same movement.

### Figures saved
- `results/figures/gradcam_sample_overview.png` — 8-sample Grad-CAM overview
- `results/figures/gradcam_correct_predictions.png` — diverse correct predictions across 8 artists
- `results/figures/gradcam_misclassifications.png` — 8 most confident wrong predictions
- `results/figures/gradcam_cross_class_comparison.png` — Monet vs Pissarro side-by-side
- `results/gradcam/` — 91 individual heatmaps (2 correct + 2 wrong per class)